In [1]:
import os
import sys

sys.path.append("../../../")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import copy
import torch
from datetime import datetime
from utils.helper import ModelConfig, color_print
from utils.dataset_utils.load_dataset import (
    load_data,
)
from utils.model_utils.load_model import load_model
from utils.model_utils.save_module import save_module
from utils.model_utils.evaluate import evaluate_model, get_sparsity, similar
from utils.dataset_utils.sampling import SamplingDataset

In [3]:
name = "bert-4-128-yahoo"
# name = "YahooAnswersTopics"
device = torch.device("cuda:0")
checkpoint = None
batch_size = 16
num_workers = 4
num_samples = 128
head_pruning_ratio = 0.5
seed = 44
include_layers = ["attention", "intermediate", "output"]
exclude_layers = None

In [4]:
model_config = ModelConfig(name, device)
num_labels = model_config.config["num_labels"]
model, tokenizer, checkpoint = load_model(model_config)

Loading the model.
{'model_name': 'Models/bert-4-128-yahoo', 'tokenizer_name': 'sadickam/sdg-classification-bert', 'task_type': 'classification', 'architectures': 'bert', 'dataset_name': 'YahooAnswersTopics', 'num_labels': 10, 'cache_dir': 'Models'}
The model Models/bert-4-128-yahoo is loaded.


In [5]:
train_dataloader, valid_dataloader, test_dataloader = load_data(
    model_config.dataset_name,
    batch_size=batch_size,
    num_workers=num_workers,
    do_cache=True,
    seed=seed,
)

Loading cached dataset YahooAnswersTopics.
train.pkl is loaded from cache.
valid.pkl is loaded from cache.
test.pkl is loaded from cache.
The dataset YahooAnswersTopics is loaded
{'dataset_name': 'YahooAnswersTopics', 'path': 'yahoo_answers_topics', 'config_name': 'yahoo_answers_topics', 'text_column': 'question_title', 'label_column': 'topic', 'cache_dir': 'Datasets/YahooAnswersTopics', 'task_type': 'classification'}


In [6]:
import numpy as np
import torch
import copy
from functools import partial
import torch.nn as nn
from transformers.pytorch_utils import find_pruneable_heads_and_indices
from typing import *

import time


def calculate_prune_head(arr, i, pruned_heads):
    flattened_with_indices = [
        (value, index)
        for index, value in np.ndenumerate(arr)
        if index not in pruned_heads
    ]

    sorted_by_value = sorted(flattened_with_indices, key=lambda x: x[0])
    bottom_indices = sorted_by_value[:i]

    bottom_indices_only = [index for _, index in bottom_indices]

    return bottom_indices_only


def prune_head(model, prune_list):
    for layer_index, head_index in prune_list:
        prune_heads(model.bert.encoder.layer[layer_index].attention, ([head_index]))
    return model


def preprocess_prunehead(arr, num_layer):
    layer_max = lambda arr: np.argmax(arr, axis=1)

    max_layer = layer_max(arr)
    for layer in range(num_layer):
        head = max_layer[layer]
        arr[layer][head] = 100
    return arr


def prune_heads(layer, heads):
    if len(heads) == 0:
        return
    heads, index = find_pruneable_heads_and_indices(
        heads,
        layer.self.num_attention_heads,
        layer.self.attention_head_size,
        layer.pruned_heads,
    )

    # Zero out weights in linear layers instead of pruning
    layer.self.query = zero_out_head_weights(
        layer.self.query, heads, layer.self.attention_head_size
    )
    layer.self.key = zero_out_head_weights(
        layer.self.key, heads, layer.self.attention_head_size
    )
    layer.self.value = zero_out_head_weights(
        layer.self.value, heads, layer.self.attention_head_size
    )
    layer.output.dense = zero_out_head_weights(
        layer.output.dense, heads, layer.self.attention_head_size, dim=1
    )


def zero_out_head_weights(
    layer: nn.Linear, heads: Set[int], head_size: int, dim: int = 0
) -> nn.Linear:
    """
    Zero out the weights of the specified heads in the linear layer.

    Args:
        layer (`torch.nn.Linear`): The layer to modify.
        heads (`Set[int]`): The indices of heads to zero out.
        head_size (`int`): The size of each head.
        dim (`int`, *optional*, defaults to 0): The dimension on which to zero out the weights.

    Returns:
        `torch.nn.Linear`: The modified layer with weights of specified heads zeroed out.
    """
    for head in heads:
        start_index = head * head_size
        end_index = (head + 1) * head_size
        if dim == 0:
            layer.weight.data[start_index:end_index] = 0
        elif dim == 1:
            layer.weight.data[:, start_index:end_index] = 0

    return layer

In [7]:
import numpy as np


def calculate_head_importance(
    model,
    model_config,
    data,
    normalize_scores_by_layer=True,
):
    device = model_config.device
    from functools import partial

    gradients = {}
    context_layers = {}

    def save_grad(gradients, layer_idx, grad):
        gradients[f"context_layer_{layer_idx}"] = grad

    def forward_hook(module, input, output, gradients, context_layers, layer_idx):
        context_layers[f"context_layer_{layer_idx}"] = output[0]
        output[0].register_hook(partial(save_grad, gradients, layer_idx))

    def reshape(tensors, shape, num_heads):
        batch_size = shape[0]
        seq_len = shape[1]
        head_dim = shape[2] // num_heads
        tensors = tensors.reshape(batch_size, seq_len, num_heads, head_dim)
        tensors = tensors.permute(0, 2, 1, 3)
        return tensors

    forward_handles = []

    for layer_idx in range(model.bert.config.num_hidden_layers):
        self_att = model.bert.encoder.layer[layer_idx].attention.self
        handle = self_att.register_forward_hook(
            partial(
                forward_hook,
                gradients=gradients,
                context_layers=context_layers,
                layer_idx=layer_idx,
            )
        )
        forward_handles.append(handle)

    """Calculate head importance scores"""
    # Disable dropout
    model.eval()
    # Device
    device = device or next(model.parameters()).device

    # Prepare data loader
    # Head importance tensor
    n_layers = model.bert.config.num_hidden_layers
    n_heads = model.bert.config.num_attention_heads
    head_dim = model.bert.config.hidden_size // n_heads
    head_importance = torch.zeros(n_layers, n_heads).to(device)
    tot_tokens = 0

    for step, batch in enumerate(data):
        input_ids = batch["input_ids"].to(device)
        input_mask = batch["attention_mask"].to(device)
        label_ids = batch["labels"].to(device)
        # Compute gradients
        loss = model(input_ids, attention_mask=input_mask, labels=label_ids).loss
        loss.backward()

        for layer_idx in range(model.bert.config.num_hidden_layers):
            ctx = context_layers[f"context_layer_{layer_idx}"]
            grad_ctx = gradients[f"context_layer_{layer_idx}"]
            shape = ctx.shape
            ctx = reshape(ctx, shape, n_heads)
            grad_ctx = reshape(ctx, shape, n_heads)

            # Take the dot
            dot = torch.einsum("bhli,bhli->bhl", [grad_ctx, ctx])
            head_importance[layer_idx] += dot.abs().sum(-1).sum(0).detach()
            del ctx, grad_ctx, dot

        tot_tokens += input_mask.float().detach().sum().data

    for layer_idx in range(model.bert.config.num_hidden_layers):
        for head in range(n_heads):
            start_idx = head * head_dim
            end_idx = (head + 1) * head_dim

            value_weight_norm = torch.norm(
                model.bert.encoder.layer[layer_idx].attention.self.value.weight[
                    :, start_idx:end_idx
                ]
            )
            head_importance[layer_idx] += value_weight_norm.detach()

    head_importance[:-1] /= tot_tokens

    # Layerwise importance normalization
    if normalize_scores_by_layer:
        exponent = 2
        norm_by_layer = torch.pow(
            torch.pow(head_importance, exponent).sum(-1), 1 / exponent
        )
        head_importance /= norm_by_layer.unsqueeze(-1) + 1e-20

    for handle in forward_handles:
        handle.remove()
    return head_importance

In [8]:
positive_samples = SamplingDataset(
    train_dataloader,
    0,
    num_samples,
    num_labels,
    True,
    4,
    device=device,
    resample=False,
    seed=seed,
)
all_samples = SamplingDataset(
    train_dataloader,
    200,
    num_samples,
    num_labels,
    False,
    4,
    device=device,
    resample=False,
    seed=seed,
)

In [9]:
def head_importance_prunning(
    model, model_config, dominant_concern, concern, sparsity_ratio, gradually=True
):
    num_attention_heads = model.config.num_attention_heads
    num_hidden_layers = model.config.num_hidden_layers
    total_heads_to_prune = int(num_attention_heads * num_hidden_layers * sparsity_ratio)

    if total_heads_to_prune >= 4 and total_heads_to_prune % 4 != 0:
        total_heads_to_prune -= 4 - (total_heads_to_prune % 4)

    if gradually:
        num_steps = max(1, total_heads_to_prune // 4)
    else:
        num_steps = 1

    heads_per_step = int(total_heads_to_prune // num_steps)
    print(f"Total heads to prune: {total_heads_to_prune}")

    pruned_heads = set()

    for step in range(num_steps):
        if step == num_steps - 1:
            current_heads_to_prune = total_heads_to_prune - (step * heads_per_step)
        else:
            current_heads_to_prune = heads_per_step

        head_importance_list = calculate_head_importance(
            model,
            model_config,
            dominant_concern,
            normalize_scores_by_layer=True,
        )
        print(f"head importance list\n {head_importance_list}")
        head_importance_list = head_importance_list.cpu()

        # preprocess_prunehead(head_importance_list, num_hidden_layers)

        prune_list = calculate_prune_head(
            head_importance_list, current_heads_to_prune, pruned_heads
        )
        pruned_heads.update(prune_list)

        prune_head(model, prune_list)
    print(pruned_heads)

In [10]:
module = copy.deepcopy(model)
head_importance_prunning(module, model_config, all_samples, 0, head_pruning_ratio)
result = evaluate_model(module, model_config, test_dataloader)
similar(
    model,
    module,
    valid_dataloader,
    0,
    num_samples,
    num_labels,
    device=device,
    seed=seed,
)

Total heads to prune: 8
head importance list
 tensor([[0.5001, 0.6139, 0.4044, 0.4577],
        [0.4735, 0.5860, 0.4591, 0.4708],
        [0.4629, 0.5403, 0.4214, 0.5623],
        [0.3790, 0.5003, 0.5684, 0.5320]], device='cuda:0')
head importance list
 tensor([[0.6176, 0.7865, 0.0031, 0.0021],
        [0.4611, 0.5914, 0.4446, 0.4899],
        [0.4920, 0.6052, 0.0049, 0.6258],
        [0.0022, 0.5444, 0.6007, 0.5855]], device='cuda:0')
{(1, 2), (0, 3), (2, 0), (3, 0), (0, 2), (2, 2), (1, 0), (1, 3)}


Evaluating the model:   0%|          | 0/1875 [00:00<?, ?it/s]

Loss: 1.3549
Precision: 0.6540, Recall: 0.5719, F1-Score: 0.5887
              precision    recall  f1-score   support

           0       0.49      0.53      0.51      2941
           1       0.67      0.37      0.48      2997
           2       0.70      0.59      0.64      3016
           3       0.27      0.71      0.39      2978
           4       0.81      0.64      0.72      3017
           5       0.88      0.68      0.77      3004
           6       0.66      0.40      0.49      3037
           7       0.55      0.67      0.61      3026
           8       0.69      0.59      0.64      2997
           9       0.82      0.54      0.65      2987

    accuracy                           0.57     30000
   macro avg       0.65      0.57      0.59     30000
weighted avg       0.65      0.57      0.59     30000

adding eps to diagonal and taking inverse
taking square root
dot products...
trying to take final svd
computed everything!
adding eps to diagonal and taking inverse
taking squa

In [11]:
for concern in range(num_labels):
    train = copy.deepcopy(train_dataloader)
    positive_samples = SamplingDataset(
        train,
        concern,
        num_samples,
        num_labels,
        True,
        4,
        device=device,
        resample=False,
        seed=seed,
    )

    module = copy.deepcopy(model)
    head_importance_prunning(
        module, model_config, positive_samples, concern, head_pruning_ratio
    )
    print(f"Evaluate the pruned model {concern}")

    result = evaluate_model(module, model_config, test_dataloader)
    similar(
        model,
        module,
        valid_dataloader,
        0,
        num_samples,
        num_labels,
        device=device,
        seed=seed,
    )

Total heads to prune: 8
head importance list
 tensor([[0.4508, 0.6332, 0.4055, 0.4811],
        [0.3895, 0.5733, 0.5322, 0.4862],
        [0.4598, 0.5867, 0.3936, 0.5381],
        [0.4054, 0.5168, 0.5721, 0.4911]], device='cuda:0')
head importance list
 tensor([[0.4891, 0.6934, 0.0035, 0.5292],
        [0.0050, 0.6118, 0.5660, 0.5526],
        [0.4713, 0.6376, 0.0061, 0.6094],
        [0.0027, 0.5742, 0.6261, 0.5275]], device='cuda:0')
{(0, 0), (0, 3), (2, 0), (3, 0), (0, 2), (3, 3), (2, 2), (1, 0)}
Evaluate the pruned model 0


Evaluating the model:   0%|          | 0/1875 [00:00<?, ?it/s]

Loss: 1.3441
Precision: 0.6535, Recall: 0.5713, F1-Score: 0.5863
              precision    recall  f1-score   support

           0       0.47      0.54      0.50      2941
           1       0.71      0.30      0.43      2997
           2       0.73      0.52      0.61      3016
           3       0.26      0.70      0.38      2978
           4       0.83      0.61      0.71      3017
           5       0.87      0.73      0.79      3004
           6       0.65      0.41      0.50      3037
           7       0.59      0.66      0.62      3026
           8       0.66      0.62      0.64      2997
           9       0.77      0.62      0.69      2987

    accuracy                           0.57     30000
   macro avg       0.65      0.57      0.59     30000
weighted avg       0.65      0.57      0.59     30000

adding eps to diagonal and taking inverse
taking square root
dot products...
trying to take final svd
computed everything!
adding eps to diagonal and taking inverse
taking squa

Evaluating the model:   0%|          | 0/1875 [00:00<?, ?it/s]

Loss: 1.3741
Precision: 0.6570, Recall: 0.5631, F1-Score: 0.5816
              precision    recall  f1-score   support

           0       0.48      0.54      0.51      2941
           1       0.69      0.35      0.46      2997
           2       0.71      0.58      0.64      3016
           3       0.26      0.73      0.38      2978
           4       0.81      0.62      0.71      3017
           5       0.90      0.67      0.77      3004
           6       0.67      0.39      0.49      3037
           7       0.55      0.67      0.61      3026
           8       0.70      0.55      0.62      2997
           9       0.81      0.53      0.64      2987

    accuracy                           0.56     30000
   macro avg       0.66      0.56      0.58     30000
weighted avg       0.66      0.56      0.58     30000

adding eps to diagonal and taking inverse
taking square root
dot products...
trying to take final svd
computed everything!
adding eps to diagonal and taking inverse
taking squa

...

Loss: 1.3256
Precision: 0.6523, Recall: 0.5859, F1-Score: 0.5976
              precision    recall  f1-score   support

           0       0.48      0.55      0.51      2941
           1       0.72      0.36      0.48      2997
           2       0.68      0.63      0.65      3016
           3       0.29      0.68      0.41      2978
           4       0.81      0.68      0.74      3017
           5       0.86      0.73      0.79      3004
           6       0.66      0.40      0.50      3037
           7       0.55      0.68      0.61      3026
           8       0.68      0.57      0.62      2997
           9       0.79      0.58      0.67      2987

    accuracy                           0.59     30000
   macro avg       0.65      0.59      0.60     30000
weighted avg       0.65      0.59      0.60     30000

adding eps to diagonal and taking inverse
taking square root
dot products...
trying to take final svd
computed everything!
adding eps to diagonal and taking inverse
taking squa

Evaluating the model:   0%|          | 0/1875 [00:00<?, ?it/s]

Loss: 1.4013
Precision: 0.6337, Recall: 0.5599, F1-Score: 0.5706
              precision    recall  f1-score   support

           0       0.45      0.54      0.49      2941
           1       0.62      0.40      0.49      2997
           2       0.72      0.52      0.60      3016
           3       0.29      0.68      0.40      2978
           4       0.83      0.55      0.66      3017
           5       0.81      0.75      0.78      3004
           6       0.68      0.36      0.47      3037
           7       0.48      0.71      0.57      3026
           8       0.73      0.44      0.55      2997
           9       0.73      0.65      0.69      2987

    accuracy                           0.56     30000
   macro avg       0.63      0.56      0.57     30000
weighted avg       0.63      0.56      0.57     30000

adding eps to diagonal and taking inverse
taking square root
dot products...
trying to take final svd
computed everything!
adding eps to diagonal and taking inverse
taking squa

Evaluating the model:   0%|          | 0/1875 [00:00<?, ?it/s]

Loss: 1.3549
Precision: 0.6540, Recall: 0.5719, F1-Score: 0.5887
              precision    recall  f1-score   support

           0       0.49      0.53      0.51      2941
           1       0.67      0.37      0.48      2997
           2       0.70      0.59      0.64      3016
           3       0.27      0.71      0.39      2978
           4       0.81      0.64      0.72      3017
           5       0.88      0.68      0.77      3004
           6       0.66      0.40      0.49      3037
           7       0.55      0.67      0.61      3026
           8       0.69      0.59      0.64      2997
           9       0.82      0.54      0.65      2987

    accuracy                           0.57     30000
   macro avg       0.65      0.57      0.59     30000
weighted avg       0.65      0.57      0.59     30000

